In [1]:
# === Imports ===
import pandas as pd
import numpy as np

# Configuration d'affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Chemins
DATA_RAW = '../data/raw/'
DATA_CLEAN = '../data/clean/'

# Chargement du dataset complet (pas de nrows cette fois !)
df = pd.read_csv(DATA_RAW + 'movie_metadata.csv')

print(f"Dataset chargé : {df.shape[0]} films, {df.shape[1]} colonnes")
print(f"Mémoire utilisée : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Dataset chargé : 5043 films, 28 colonnes
Mémoire utilisée : 4.53 MB


In [2]:
# === Repérage des doublons ===
nb_doublons = df.duplicated().sum()
print(f"Nombre de lignes en doublon parfait : {nb_doublons}")

# Doublons sur le titre (un même film qui apparaîtrait 2 fois)
nb_doublons_titre = df.duplicated(subset=['movie_title']).sum()
print(f"Nombre de doublons sur le titre : {nb_doublons_titre}")

Nombre de lignes en doublon parfait : 45
Nombre de doublons sur le titre : 126


In [3]:
# === Suppression des doublons ===
nb_avant = len(df)

# On supprime les doublons parfaits (toutes colonnes identiques)
df = df.drop_duplicates()

# On supprime aussi les doublons sur le titre + année (même film, même sortie)
df = df.drop_duplicates(subset=['movie_title', 'title_year'])

nb_apres = len(df)
print(f"Avant : {nb_avant} films")
print(f"Après : {nb_apres} films")
print(f"Supprimés : {nb_avant - nb_apres} doublons")

Avant : 5043 films
Après : 4919 films
Supprimés : 124 doublons


In [4]:
# === Suppression des lignes avec infos critiques manquantes ===
nb_avant = len(df)

# On garde uniquement les films qui ont titre + genres + année
df = df.dropna(subset=['movie_title', 'genres', 'title_year'])

nb_apres = len(df)
print(f"Avant : {nb_avant} films")
print(f"Après : {nb_apres} films")
print(f"Supprimés : {nb_avant - nb_apres} films sans infos critiques")

Avant : 4919 films
Après : 4813 films
Supprimés : 106 films sans infos critiques


In [5]:
# === Conversion des types ===
# title_year en entier (plus propre que 2009.0)
df['title_year'] = df['title_year'].astype(int)

# duration en entier également
df['duration'] = df['duration'].fillna(0).astype(int)

print("Types corrigés :")
print(df[['title_year', 'duration', 'imdb_score']].dtypes)
print()
print("Aperçu :")
df[['movie_title', 'title_year', 'duration', 'imdb_score']].head()

Types corrigés :
title_year      int64
duration        int64
imdb_score    float64
dtype: object

Aperçu :


,movie_title,title_year,duration,imdb_score
0,Avatar,2009,178,7.9
1,Pirates of the Caribbean: At World's End,2007,169,7.1
2,Spectre,2015,148,6.8
3,The Dark Knight Rises,2012,164,8.5
5,John Carter,2012,132,6.6


In [6]:
# === Analyse de la colonne genres ===

# Compter le nombre de genres par film
df['nb_genres'] = df['genres'].str.count(r'\|') + 1

print("Distribution du nombre de genres par film :")
print(df['nb_genres'].value_counts().sort_index())
print(f"\nNombre moyen de genres : {df['nb_genres'].mean():.2f}")
print(f"Nombre max de genres : {df['nb_genres'].max()}")

Distribution du nombre de genres par film :
nb_genres
1     607
2    1305
3    1545
4     931
5     333
6      70
7      18
8       4
Name: count, dtype: int64

Nombre moyen de genres : 2.87
Nombre max de genres : 8


In [7]:
# === Liste de tous les genres uniques ===

# On split tous les genres et on les met dans une liste plate
tous_genres = df['genres'].str.split('|').explode().unique()

print(f"Nombre de genres distincts : {len(tous_genres)}")
print(f"\nListe des genres :")
print(sorted(tous_genres))

Nombre de genres distincts : 24

Liste des genres :
['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War', 'Western']


In [8]:
# === Split des genres en plusieurs colonnes ===

# split sur le caractère | et expansion en colonnes
genres_split = df['genres'].str.split('|', expand=True)

# Renommer les colonnes proprement
genres_split.columns = [f'genre_{i+1}' for i in range(genres_split.shape[1])]

# Combien de colonnes créées ?
print(f"Nombre de colonnes créées : {genres_split.shape[1]}")
print(f"\nAperçu :")
genres_split.head()

Nombre de colonnes créées : 8

Aperçu :


,genre_1,genre_2,genre_3,genre_4,genre_5,genre_6,genre_7,genre_8
0,Action,Adventure,Fantasy,Sci-Fi,None,None,None,None
1,Action,Adventure,Fantasy,None,None,None,None,None
2,Action,Adventure,Thriller,None,None,None,None,None
3,Action,Thriller,None,None,None,None,None,None
5,Action,Adventure,Sci-Fi,None,None,None,None,None


In [9]:
# === Pertinence des colonnes genre_n ===

print("Pourcentage de films ayant un genre dans chaque colonne :")
for col in genres_split.columns:
    pct = genres_split[col].notna().sum() / len(genres_split) * 100
    print(f"  {col} : {pct:.1f}%")

Pourcentage de films ayant un genre dans chaque colonne :
  genre_1 : 100.0%
  genre_2 : 87.4%
  genre_3 : 60.3%
  genre_4 : 28.2%
  genre_5 : 8.8%
  genre_6 : 1.9%
  genre_7 : 0.5%
  genre_8 : 0.1%


In [10]:
# === Garder seulement genre_1, genre_2, genre_3 et les ajouter à df ===

# On garde les 3 premières colonnes
df['genre_1'] = genres_split['genre_1']
df['genre_2'] = genres_split['genre_2']
df['genre_3'] = genres_split['genre_3']

# Vérification
print("Aperçu des nouvelles colonnes :")
df[['movie_title', 'genre_1', 'genre_2', 'genre_3']].head(10)

Aperçu des nouvelles colonnes :


,movie_title,genre_1,genre_2,genre_3
0,Avatar,Action,Adventure,Fantasy
1,Pirates of the Caribbean: At World's End,Action,Adventure,Fantasy
2,Spectre,Action,Adventure,Thriller
3,The Dark Knight Rises,Action,Thriller,None
5,John Carter,Action,Adventure,Sci-Fi
6,Spider-Man 3,Action,Adventure,Romance
7,Tangled,Adventure,Animation,Comedy
8,Avengers: Age of Ultron,Action,Adventure,Sci-Fi
9,Harry Potter and the Half-Blood Prince,Adventure,Family,Fantasy
10,Batman v Superman: Dawn of Justice,Action,Adventure,Sci-Fi


In [11]:
# === Extraction de l'ID IMDb depuis l'URL ===

# Aperçu d'une URL pour vérifier le format
print("Exemple d'URL :")
print(df['movie_imdb_link'].iloc[0])
print()

# Extraction avec une expression régulière (regex)
# On cherche le motif "tt" suivi de chiffres
df['imdb_id'] = df['movie_imdb_link'].str.extract(r'(tt\d+)')

# Vérification
print("Exemples d'IDs extraits :")
print(df[['movie_title', 'imdb_id']].head(5))
print()

# Combien d'extractions réussies ?
nb_ok = df['imdb_id'].notna().sum()
print(f"IDs extraits avec succès : {nb_ok} / {len(df)} ({nb_ok/len(df)*100:.1f}%)")

Exemple d'URL :
http://www.imdb.com/title/tt0499549/?ref_=fn_tt_tt_1

Exemples d'IDs extraits :
                                 movie_title    imdb_id
0                                    Avatar   tt0499549
1  Pirates of the Caribbean: At World's End   tt0449088
2                                   Spectre   tt2379713
3                     The Dark Knight Rises   tt1345836
5                               John Carter   tt0401729

IDs extraits avec succès : 4813 / 4813 (100.0%)


In [12]:
# === Premier appel à l'API OMDb ===

import requests  # bibliothèque pour faire des requêtes HTTP

API_KEY = 'ma_cle_api'

# URL de base de l'API OMDb
BASE_URL = 'http://www.omdbapi.com/'

print("Setup OK ✅")

Setup OK ✅


In [13]:
# === Test : récupérer les infos d'Avatar ===

# Paramètres de la requête
params = {
    'apikey': API_KEY,
    'i': 'tt0499549'  # i = recherche par IMDb ID
}

# Appel à l'API
response = requests.get(BASE_URL, params=params)

# Conversion de la réponse en JSON (= dictionnaire Python)
data = response.json()

# Affichage propre
print("Statut HTTP :", response.status_code)
print()
print("Données reçues :")
for cle, valeur in data.items():
    print(f"  {cle} : {valeur}")

Statut HTTP : 200

Données reçues :
  Title : Avatar
  Year : 2009
  Rated : PG-13
  Released : 18 Dec 2009
  Runtime : 162 min
  Genre : Action, Adventure, Fantasy
  Director : James Cameron
  Writer : James Cameron
  Actors : Sam Worthington, Zoe Saldaña, Sigourney Weaver
  Plot : A paraplegic Marine dispatched to the moon Pandora on a unique mission becomes torn between following his orders and protecting the world he feels is his home.
  Language : English, Spanish
  Country : United States, United Kingdom
  Awards : Won 3 Oscars. 91 wins & 131 nominations total
  Poster : https://m.media-amazon.com/images/M/MV5BMDEzMmQwZjctZWU2My00MWNlLWE0NjItMDJlYTRlNGJiZjcyXkEyXkFqcGc@._V1_SX300.jpg
  Ratings : [{'Source': 'Internet Movie Database', 'Value': '7.9/10'}, {'Source': 'Rotten Tomatoes', 'Value': '81%'}, {'Source': 'Metacritic', 'Value': '83/100'}]
  Metascore : 83
  imdbRating : 7.9
  imdbVotes : 1,486,308
  imdbID : tt0499549
  Type : movie
  DVD : N/A
  BoxOffice : $785,221,649
  P

In [14]:
# === Fonction réutilisable pour récupérer les données d'un film ===

def get_omdb_data(imdb_id, api_key=API_KEY):
    """
    Récupère les données enrichies d'un film via l'API OMDb.
    Retourne un dictionnaire ou None en cas d'erreur.
    """
    params = {
        'apikey': api_key,
        'i': imdb_id
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        data = response.json()
        if data.get('Response') == 'True':
            return data
        else:
            return None
    except Exception as e:
        print(f"Erreur pour {imdb_id} : {e}")
        return None

# Test rapide sur Avatar
test = get_omdb_data('tt0499549')
print("Test fonction OK :", test['Title'] if test else "ÉCHEC")

Test fonction OK : Avatar


In [15]:
# === Sélection des 500 films les plus populaires ===

# Tri par nombre de votes décroissant et prise des 500 premiers
df_sample = df.nlargest(500, 'num_voted_users').copy().reset_index(drop=True)

print(f"Échantillon créé : {len(df_sample)} films")
print(f"\nNombre de votes min : {df_sample['num_voted_users'].min():,}")
print(f"Nombre de votes max : {df_sample['num_voted_users'].max():,}")
print(f"\nTop 10 :")
df_sample[['movie_title', 'title_year', 'imdb_score', 'num_voted_users']].head(10)

Échantillon créé : 500 films

Nombre de votes min : 212,167
Nombre de votes max : 1,689,764

Top 10 :


,movie_title,title_year,imdb_score,num_voted_users
0,The Shawshank Redemption,1994,9.3,1689764
1,The Dark Knight,2008,9.0,1676169
2,Inception,2010,8.8,1468200
3,Fight Club,1999,8.8,1347461
4,Pulp Fiction,1994,8.9,1324680
5,Forrest Gump,1994,8.8,1251222
6,The Lord of the Rings: The Fellowship of the R...,2001,8.8,1238746
7,The Matrix,1999,8.7,1217752
8,The Lord of the Rings: The Return of the King,2003,8.9,1215718
9,The Godfather,1972,9.2,1155770


In [16]:
# Installation de tqdm pour la barre de progression
!pip install tqdm

In [17]:
# === Enrichissement des 500 films via l'API OMDb ===

from tqdm.notebook import tqdm
import time

# Liste pour stocker tous les résultats
omdb_data = []

# Boucle sur chaque film de l'échantillon
print(f"Lancement de l'enrichissement pour {len(df_sample)} films...")
print("Estimation : 4-6 minutes\n")

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Enrichissement"):
    imdb_id = row['imdb_id']
    
    # Appel API
    data = get_omdb_data(imdb_id)
    
    if data is not None:
        # On stocke uniquement les colonnes utiles pour le projet
        omdb_data.append({
            'imdb_id': imdb_id,
            'omdb_title': data.get('Title'),
            'omdb_year': data.get('Year'),
            'rated': data.get('Rated'),
            'released': data.get('Released'),
            'runtime': data.get('Runtime'),
            'omdb_genre': data.get('Genre'),
            'director': data.get('Director'),
            'writer': data.get('Writer'),
            'actors': data.get('Actors'),
            'plot': data.get('Plot'),
            'language': data.get('Language'),
            'country': data.get('Country'),
            'awards': data.get('Awards'),
            'poster': data.get('Poster'),
            'metascore': data.get('Metascore'),
            'omdb_imdb_rating': data.get('imdbRating'),
            'omdb_imdb_votes': data.get('imdbVotes'),
            'box_office': data.get('BoxOffice')
        })
    else:
        # En cas d'erreur on garde au moins l'ID pour ne pas perdre l'index
        omdb_data.append({'imdb_id': imdb_id})

print(f"\n✅ Terminé ! {len(omdb_data)} films traités")

Lancement de l'enrichissement pour 500 films...
Estimation : 4-6 minutes



Enrichissement:   0%|          | 0/500 [00:00<?, ?it/s]


✅ Terminé ! 500 films traités


In [18]:
# === Conversion de la liste OMDb en DataFrame ===

df_omdb = pd.DataFrame(omdb_data)

print(f"Dimensions : {df_omdb.shape}")
print(f"\nColonnes :")
print(df_omdb.columns.tolist())
print(f"\nAperçu :")
df_omdb.head(3)

Dimensions : (500, 19)

Colonnes :
['imdb_id', 'omdb_title', 'omdb_year', 'rated', 'released', 'runtime', 'omdb_genre', 'director', 'writer', 'actors', 'plot', 'language', 'country', 'awards', 'poster', 'metascore', 'omdb_imdb_rating', 'omdb_imdb_votes', 'box_office']

Aperçu :


,imdb_id,omdb_title,omdb_year,rated,released,runtime,omdb_genre,director,writer,actors,plot,language,country,awards,poster,metascore,omdb_imdb_rating,omdb_imdb_votes,box_office
0,tt0111161,The Shawshank Redemption,1994,R,14 Oct 1994,142 min,Drama,Frank Darabont,"Stephen King, Frank Darabont","Tim Robbins, Morgan Freeman, Bob Gunton",A wrongfully convicted banker forms a close fr...,English,United States,Nominated for 7 Oscars. 21 wins & 42 nominatio...,https://m.media-amazon.com/images/M/MV5BMDAyY2...,82,9.3,"3,182,645","$28,767,189"
1,tt0468569,The Dark Knight,2008,PG-13,18 Jul 2008,152 min,"Action, Crime, Drama",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Goyer","Christian Bale, Heath Ledger, Aaron Eckhart",When a menace known as the Joker wreaks havoc ...,"English, Mandarin","United States, United Kingdom",Won 2 Oscars. 163 wins & 165 nominations total,https://m.media-amazon.com/images/M/MV5BMTMxNT...,85,9.1,"3,161,907","$534,987,076"
2,tt1375666,Inception,2010,PG-13,16 Jul 2010,148 min,"Action, Adventure, Sci-Fi",Christopher Nolan,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",A thief who steals corporate secrets through t...,"English, Japanese, French","United Kingdom, United States",Won 4 Oscars. 160 wins & 220 nominations total,https://m.media-amazon.com/images/M/MV5BMjAxMz...,74,8.8,"2,811,614","$292,587,330"


In [19]:
# === Fusion des deux datasets sur imdb_id ===

df_final = df_sample.merge(df_omdb, on='imdb_id', how='left')

print(f"Dataset final : {df_final.shape[0]} films, {df_final.shape[1]} colonnes")
print(f"\nQuelques nouvelles colonnes enrichies :")
df_final[['movie_title', 'director', 'actors', 'plot', 'poster']].head(3)

Dataset final : 500 films, 51 colonnes

Quelques nouvelles colonnes enrichies :


,movie_title,director,actors,plot,poster
0,The Shawshank Redemption,Frank Darabont,"Tim Robbins, Morgan Freeman, Bob Gunton",A wrongfully convicted banker forms a close fr...,https://m.media-amazon.com/images/M/MV5BMDAyY2...
1,The Dark Knight,Christopher Nolan,"Christian Bale, Heath Ledger, Aaron Eckhart",When a menace known as the Joker wreaks havoc ...,https://m.media-amazon.com/images/M/MV5BMTMxNT...
2,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",A thief who steals corporate secrets through t...,https://m.media-amazon.com/images/M/MV5BMjAxMz...


In [20]:
# === Vérification des données enrichies ===

print("Pourcentage de remplissage des nouvelles colonnes OMDb :")
colonnes_omdb = ['plot', 'poster', 'director', 'actors', 'awards', 'box_office', 'metascore']
for col in colonnes_omdb:
    pct = df_final[col].notna().sum() / len(df_final) * 100
    print(f"  {col} : {pct:.1f}%")

Pourcentage de remplissage des nouvelles colonnes OMDb :
  plot : 100.0%
  poster : 100.0%
  director : 100.0%
  actors : 100.0%
  awards : 100.0%
  box_office : 100.0%
  metascore : 100.0%


In [21]:
# === Sauvegarde du dataset nettoyé et enrichi ===

# 1. Le dataset complet nettoyé (4813 films IMDb sans enrichissement OMDb)
df.to_csv(DATA_CLEAN + 'imdb_5000_clean.csv', index=False)
print(f"✅ Sauvegardé : imdb_5000_clean.csv ({len(df)} films, {df.shape[1]} colonnes)")

# 2. Le dataset enrichi (500 films avec données OMDb pour le ML)
df_final.to_csv(DATA_CLEAN + 'imdb_500_enriched.csv', index=False)
print(f"✅ Sauvegardé : imdb_500_enriched.csv ({len(df_final)} films, {df_final.shape[1]} colonnes)")

print(f"\nFichiers sauvegardés dans : {DATA_CLEAN}")

✅ Sauvegardé : imdb_5000_clean.csv (4813 films, 33 colonnes)
✅ Sauvegardé : imdb_500_enriched.csv (500 films, 51 colonnes)

Fichiers sauvegardés dans : ../data/clean/


In [22]:
# === Élargir l'enrichissement à 1000 films au total ===

# On reprend les 1000 films les plus populaires
df_top1000 = df.nlargest(1000, 'num_voted_users').copy().reset_index(drop=True)

# On retire ceux qu'on a déjà enrichis (les 500 de df_sample)
ids_deja_faits = set(df_sample['imdb_id'])
df_nouveaux = df_top1000[~df_top1000['imdb_id'].isin(ids_deja_faits)].copy().reset_index(drop=True)

print(f"Films déjà enrichis : {len(ids_deja_faits)}")
print(f"Nouveaux films à enrichir : {len(df_nouveaux)}")
print(f"Total visé : 1000")

Films déjà enrichis : 500
Nouveaux films à enrichir : 500
Total visé : 1000


In [23]:
# === Enrichissement des 500 nouveaux films ===

omdb_data_2 = []

print(f"Lancement pour {len(df_nouveaux)} nouveaux films...")

for idx, row in tqdm(df_nouveaux.iterrows(), total=len(df_nouveaux), desc="Enrichissement v2"):
    imdb_id = row['imdb_id']
    data = get_omdb_data(imdb_id)
    
    if data is not None:
        omdb_data_2.append({
            'imdb_id': imdb_id,
            'omdb_title': data.get('Title'),
            'omdb_year': data.get('Year'),
            'rated': data.get('Rated'),
            'released': data.get('Released'),
            'runtime': data.get('Runtime'),
            'omdb_genre': data.get('Genre'),
            'director': data.get('Director'),
            'writer': data.get('Writer'),
            'actors': data.get('Actors'),
            'plot': data.get('Plot'),
            'language': data.get('Language'),
            'country': data.get('Country'),
            'awards': data.get('Awards'),
            'poster': data.get('Poster'),
            'metascore': data.get('Metascore'),
            'omdb_imdb_rating': data.get('imdbRating'),
            'omdb_imdb_votes': data.get('imdbVotes'),
            'box_office': data.get('BoxOffice')
        })
    else:
        omdb_data_2.append({'imdb_id': imdb_id})

print(f"\n✅ Terminé ! {len(omdb_data_2)} nouveaux films traités")

Lancement pour 500 nouveaux films...


Enrichissement v2:   0%|          | 0/500 [00:00<?, ?it/s]


✅ Terminé ! 500 nouveaux films traités


In [24]:
# === Création du dataset final 1000 films ===

# Conversion des nouvelles données OMDb en DataFrame
df_omdb_2 = pd.DataFrame(omdb_data_2)

# Fusion des données IMDb (df_top1000) avec TOUTES les données OMDb (les 500 anciennes + 500 nouvelles)
df_omdb_total = pd.concat([df_omdb, df_omdb_2], ignore_index=True)

# Merge final
df_final_1000 = df_top1000.merge(df_omdb_total, on='imdb_id', how='left')

print(f"Dataset final 1000 films : {df_final_1000.shape}")

# Vérification du remplissage
print(f"\nRemplissage des colonnes OMDb :")
for col in ['plot', 'poster', 'director', 'actors']:
    pct = df_final_1000[col].notna().sum() / len(df_final_1000) * 100
    print(f"  {col} : {pct:.1f}%")

# Sauvegarde
df_final_1000.to_csv(DATA_CLEAN + 'imdb_1000_enriched.csv', index=False)
print(f"\n✅ Sauvegardé : imdb_1000_enriched.csv")

Dataset final 1000 films : (1000, 51)

Remplissage des colonnes OMDb :
  plot : 99.8%
  poster : 99.8%
  director : 99.8%
  actors : 99.8%

✅ Sauvegardé : imdb_1000_enriched.csv


In [1]:
# === Reprise pour enrichissement de 1000 films supplémentaires ===

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Configuration
DATA_CLEAN = '../data/clean/'
API_KEY = 'ma_cle_api'  # ⚠️ Remplace par ta vraie clé
BASE_URL = 'http://www.omdbapi.com/'

# Charger le dataset complet (4813 films)
df_clean = pd.read_csv(DATA_CLEAN + 'imdb_5000_clean.csv')

# Charger le dataset enrichi actuel (1000 films)
df_enriched_actuel = pd.read_csv(DATA_CLEAN + 'imdb_1000_enriched.csv')

print(f"Dataset complet : {len(df_clean)} films")
print(f"Dataset enrichi actuel : {len(df_enriched_actuel)} films")

# Identifier les films déjà enrichis
ids_deja_enrichis = set(df_enriched_actuel['imdb_id'])
print(f"Films déjà enrichis : {len(ids_deja_enrichis)}")

Dataset complet : 4813 films
Dataset enrichi actuel : 1000 films
Films déjà enrichis : 1000


In [2]:
# === Sélectionner les films 1001 à 2000 par popularité ===

# Tri du dataset complet par nombre de votes (popularité)
df_top2000 = df_clean.nlargest(2000, 'num_voted_users').copy()

# On garde uniquement ceux qu'on n'a PAS déjà enrichis
df_nouveaux = df_top2000[~df_top2000['imdb_id'].isin(ids_deja_enrichis)].copy().reset_index(drop=True)

print(f"Nouveaux films à enrichir : {len(df_nouveaux)}")
print(f"\nAperçu (les 5 premiers) :")
df_nouveaux[['movie_title', 'title_year', 'imdb_score', 'num_voted_users']].head()

Nouveaux films à enrichir : 1000

Aperçu (les 5 premiers) :


,movie_title,title_year,imdb_score,num_voted_users
0,47 Ronin,2013,6.3,116994
1,The Virgin Suicides,1999,7.2,116910
2,She's the Man,2006,6.4,116762
3,Marley & Me,2008,7.1,116681
4,Dr. No,1962,7.3,116642


In [3]:
# === Fonction d'enrichissement OMDb ===

def get_omdb_data(imdb_id, api_key=API_KEY):
    params = {'apikey': api_key, 'i': imdb_id}
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        data = response.json()
        if data.get('Response') == 'True':
            return data
        else:
            return None
    except Exception as e:
        print(f"Erreur pour {imdb_id} : {e}")
        return None

# Test rapide
test = get_omdb_data(df_nouveaux['imdb_id'].iloc[0])
print("Test OK :", test['Title'] if test else "ÉCHEC")

Test OK : 47 Ronin


In [4]:
# === Enrichissement des 1000 nouveaux films ===

omdb_data_3 = []

print(f"Lancement pour {len(df_nouveaux)} nouveaux films...")

for idx, row in tqdm(df_nouveaux.iterrows(), total=len(df_nouveaux), desc="Enrichissement v3"):
    imdb_id = row['imdb_id']
    data = get_omdb_data(imdb_id)
    
    if data is not None:
        omdb_data_3.append({
            'imdb_id': imdb_id,
            'omdb_title': data.get('Title'),
            'omdb_year': data.get('Year'),
            'rated': data.get('Rated'),
            'released': data.get('Released'),
            'runtime': data.get('Runtime'),
            'omdb_genre': data.get('Genre'),
            'director': data.get('Director'),
            'writer': data.get('Writer'),
            'actors': data.get('Actors'),
            'plot': data.get('Plot'),
            'language': data.get('Language'),
            'country': data.get('Country'),
            'awards': data.get('Awards'),
            'poster': data.get('Poster'),
            'metascore': data.get('Metascore'),
            'omdb_imdb_rating': data.get('imdbRating'),
            'omdb_imdb_votes': data.get('imdbVotes'),
            'box_office': data.get('BoxOffice')
        })
    else:
        omdb_data_3.append({'imdb_id': imdb_id})

print(f"\n✅ Terminé ! {len(omdb_data_3)} nouveaux films traités")

Lancement pour 1000 nouveaux films...


Enrichissement v3:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Terminé ! 1000 nouveaux films traités


In [8]:
# === Sauvegarde du dataset à 2000 films (version adaptative) ===

import os

# Convertir les nouvelles données OMDb en DataFrame
df_omdb_3 = pd.DataFrame(omdb_data_3)

# Étape 1 : enrichir les nouveaux films comme on l'avait fait en semaine 3
df_nouveaux_enrichis = df_nouveaux.merge(df_omdb_3, on='imdb_id', how='left')

print(f"Nouveaux films enrichis : {len(df_nouveaux_enrichis)} lignes, {df_nouveaux_enrichis.shape[1]} colonnes")
print(f"Dataset existant : {len(df_enriched_actuel)} lignes, {df_enriched_actuel.shape[1]} colonnes")

# Étape 2 : concaténer les deux datasets enrichis
df_final_2000 = pd.concat([df_enriched_actuel, df_nouveaux_enrichis], ignore_index=True)

# Supprimer les éventuels doublons
df_final_2000 = df_final_2000.drop_duplicates(subset=['imdb_id']).reset_index(drop=True)

print(f"\n📊 Dataset final 2000 films : {df_final_2000.shape}")

# Vérification du remplissage
print(f"\nRemplissage des colonnes OMDb :")
for col in ['plot', 'poster', 'director', 'actors']:
    if col in df_final_2000.columns:
        pct = df_final_2000[col].notna().sum() / len(df_final_2000) * 100
        print(f"  {col} : {pct:.1f}%")
    else:
        print(f"  {col} : COLONNE INTROUVABLE")

# Sauvegarder
df_final_2000.to_csv(DATA_CLEAN + 'imdb_2000_enriched.csv', index=False)
print(f"\n✅ Sauvegardé : imdb_2000_enriched.csv")

Nouveaux films enrichis : 1000 lignes, 51 colonnes
Dataset existant : 1000 lignes, 51 colonnes

📊 Dataset final 2000 films : (2000, 51)

Remplissage des colonnes OMDb :
  plot : 99.8%
  poster : 99.8%
  director : 99.8%
  actors : 99.8%

✅ Sauvegardé : imdb_2000_enriched.csv


In [2]:
# === Reprise pour passer de 2000 à 3000 films ===

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Configuration
DATA_CLEAN = '../data/clean/'
API_KEY = 'ma_cle_api'  #
BASE_URL = 'http://www.omdbapi.com/'

# Charger le dataset complet (4813 films)
df_clean = pd.read_csv(DATA_CLEAN + 'imdb_5000_clean.csv')

# Charger le dataset enrichi actuel (2000 films d'hier)
df_enriched_actuel = pd.read_csv(DATA_CLEAN + 'imdb_2000_enriched.csv')

print(f"Dataset complet : {len(df_clean)} films")
print(f"Dataset enrichi actuel : {len(df_enriched_actuel)} films")

ids_deja_enrichis = set(df_enriched_actuel['imdb_id'])
print(f"Films déjà enrichis : {len(ids_deja_enrichis)}")

Dataset complet : 4813 films
Dataset enrichi actuel : 2000 films
Films déjà enrichis : 2000


In [3]:
# === Sélectionner les films 2001 à 3000 par popularité ===

df_top3000 = df_clean.nlargest(3000, 'num_voted_users').copy()
df_nouveaux = df_top3000[~df_top3000['imdb_id'].isin(ids_deja_enrichis)].copy().reset_index(drop=True)

print(f"Nouveaux films à enrichir : {len(df_nouveaux)}")
print(f"\nAperçu (les 5 premiers) :")
df_nouveaux[['movie_title', 'title_year', 'imdb_score', 'num_voted_users']].head()

Nouveaux films à enrichir : 1000

Aperçu (les 5 premiers) :


,movie_title,title_year,imdb_score,num_voted_users
0,K-19: The Widowmaker,2002,6.7,49311
1,Proof of Life,2000,6.2,49300
2,Enough Said,2013,7.1,49240
3,Paddington,2014,7.2,49207
4,Larry Crowne,2011,6.1,49205


In [4]:
# === Fonction d'enrichissement OMDb ===

def get_omdb_data(imdb_id, api_key=API_KEY):
    params = {'apikey': api_key, 'i': imdb_id}
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        data = response.json()
        if data.get('Response') == 'True':
            return data
        else:
            return None
    except Exception as e:
        print(f"Erreur pour {imdb_id} : {e}")
        return None

# Test rapide
test = get_omdb_data(df_nouveaux['imdb_id'].iloc[0])
print("Test OK :", test['Title'] if test else "ÉCHEC")

Test OK : K-19: The Widowmaker


In [5]:
# === Enrichissement des 1000 nouveaux films ===

omdb_data_4 = []

print(f"Lancement pour {len(df_nouveaux)} nouveaux films...")

for idx, row in tqdm(df_nouveaux.iterrows(), total=len(df_nouveaux), desc="Enrichissement v4"):
    imdb_id = row['imdb_id']
    data = get_omdb_data(imdb_id)
    
    if data is not None:
        omdb_data_4.append({
            'imdb_id': imdb_id,
            'omdb_title': data.get('Title'),
            'omdb_year': data.get('Year'),
            'rated': data.get('Rated'),
            'released': data.get('Released'),
            'runtime': data.get('Runtime'),
            'omdb_genre': data.get('Genre'),
            'director': data.get('Director'),
            'writer': data.get('Writer'),
            'actors': data.get('Actors'),
            'plot': data.get('Plot'),
            'language': data.get('Language'),
            'country': data.get('Country'),
            'awards': data.get('Awards'),
            'poster': data.get('Poster'),
            'metascore': data.get('Metascore'),
            'omdb_imdb_rating': data.get('imdbRating'),
            'omdb_imdb_votes': data.get('imdbVotes'),
            'box_office': data.get('BoxOffice')
        })
    else:
        omdb_data_4.append({'imdb_id': imdb_id})

print(f"\n✅ Terminé ! {len(omdb_data_4)} nouveaux films traités")

Lancement pour 1000 nouveaux films...


Enrichissement v4:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Terminé ! 1000 nouveaux films traités


In [6]:
# === Sauvegarde du dataset à 3000 films ===

# Convertir les nouvelles données OMDb en DataFrame
df_omdb_4 = pd.DataFrame(omdb_data_4)

# Enrichir les nouveaux films
df_nouveaux_enrichis = df_nouveaux.merge(df_omdb_4, on='imdb_id', how='left')

print(f"Nouveaux films enrichis : {len(df_nouveaux_enrichis)} lignes, {df_nouveaux_enrichis.shape[1]} colonnes")
print(f"Dataset existant : {len(df_enriched_actuel)} lignes, {df_enriched_actuel.shape[1]} colonnes")

# Concaténer
df_final_3000 = pd.concat([df_enriched_actuel, df_nouveaux_enrichis], ignore_index=True)
df_final_3000 = df_final_3000.drop_duplicates(subset=['imdb_id']).reset_index(drop=True)

print(f"\n📊 Dataset final 3000 films : {df_final_3000.shape}")

# Vérification
print(f"\nRemplissage des colonnes OMDb :")
for col in ['plot', 'poster', 'director', 'actors']:
    if col in df_final_3000.columns:
        pct = df_final_3000[col].notna().sum() / len(df_final_3000) * 100
        print(f"  {col} : {pct:.1f}%")
    else:
        print(f"  {col} : COLONNE INTROUVABLE")

# Sauvegarder
df_final_3000.to_csv(DATA_CLEAN + 'imdb_3000_enriched.csv', index=False)
print(f"\n✅ Sauvegardé : imdb_3000_enriched.csv")

Nouveaux films enrichis : 1000 lignes, 51 colonnes
Dataset existant : 2000 lignes, 51 colonnes

📊 Dataset final 3000 films : (3000, 51)

Remplissage des colonnes OMDb :
  plot : 99.8%
  poster : 99.8%
  director : 99.8%
  actors : 99.8%

✅ Sauvegardé : imdb_3000_enriched.csv


In [3]:
# === Reprise pour passer de 3000 à 4000 films (DERNIÈRE ÉTAPE) ===

import pandas as pd
import requests
from tqdm.notebook import tqdm

# Configuration
DATA_CLEAN = '../data/clean/'
API_KEY = 'ma_cle_api'  # ⚠️ Remplace par ta vraie clé
BASE_URL = 'http://www.omdbapi.com/'

# Charger le dataset complet (4813 films)
df_clean = pd.read_csv(DATA_CLEAN + 'imdb_5000_clean.csv')

# Charger le dataset enrichi actuel (3000 films d'hier)
df_enriched_actuel = pd.read_csv(DATA_CLEAN + 'imdb_3000_enriched.csv')

print(f"Dataset complet : {len(df_clean)} films")
print(f"Dataset enrichi actuel : {len(df_enriched_actuel)} films")

ids_deja_enrichis = set(df_enriched_actuel['imdb_id'])
print(f"Films déjà enrichis : {len(ids_deja_enrichis)}")

Dataset complet : 4813 films
Dataset enrichi actuel : 3000 films
Films déjà enrichis : 3000


In [4]:
# === Sélectionner les films 3001 à 4000 par popularité ===

df_top4000 = df_clean.nlargest(4000, 'num_voted_users').copy()
df_nouveaux = df_top4000[~df_top4000['imdb_id'].isin(ids_deja_enrichis)].copy().reset_index(drop=True)

print(f"Nouveaux films à enrichir : {len(df_nouveaux)}")
print(f"\nAperçu (les 5 premiers) :")
df_nouveaux[['movie_title', 'title_year', 'imdb_score', 'num_voted_users']].head()

Nouveaux films à enrichir : 1000

Aperçu (les 5 premiers) :


,movie_title,title_year,imdb_score,num_voted_users
0,Selena,1997,6.7,19126
1,Love in the Time of Cholera,2007,6.4,19114
2,Down to Earth,2001,5.4,19079
3,In Cold Blood,1967,8.0,19026
4,The Upside of Anger,2005,6.9,19007


In [5]:
# === Fonction d'enrichissement OMDb ===

def get_omdb_data(imdb_id, api_key=API_KEY):
    params = {'apikey': api_key, 'i': imdb_id}
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        data = response.json()
        if data.get('Response') == 'True':
            return data
        else:
            return None
    except Exception as e:
        print(f"Erreur pour {imdb_id} : {e}")
        return None

# Test rapide
test = get_omdb_data(df_nouveaux['imdb_id'].iloc[0])
print("Test OK :", test['Title'] if test else "ÉCHEC")

Test OK : Selena


In [6]:
# === Enrichissement des 1000 derniers films ===

omdb_data_5 = []

print(f"Lancement pour {len(df_nouveaux)} nouveaux films...")

for idx, row in tqdm(df_nouveaux.iterrows(), total=len(df_nouveaux), desc="Enrichissement v5"):
    imdb_id = row['imdb_id']
    data = get_omdb_data(imdb_id)
    
    if data is not None:
        omdb_data_5.append({
            'imdb_id': imdb_id,
            'omdb_title': data.get('Title'),
            'omdb_year': data.get('Year'),
            'rated': data.get('Rated'),
            'released': data.get('Released'),
            'runtime': data.get('Runtime'),
            'omdb_genre': data.get('Genre'),
            'director': data.get('Director'),
            'writer': data.get('Writer'),
            'actors': data.get('Actors'),
            'plot': data.get('Plot'),
            'language': data.get('Language'),
            'country': data.get('Country'),
            'awards': data.get('Awards'),
            'poster': data.get('Poster'),
            'metascore': data.get('Metascore'),
            'omdb_imdb_rating': data.get('imdbRating'),
            'omdb_imdb_votes': data.get('imdbVotes'),
            'box_office': data.get('BoxOffice')
        })
    else:
        omdb_data_5.append({'imdb_id': imdb_id})

print(f"\n✅ Terminé ! {len(omdb_data_5)} nouveaux films traités")

Lancement pour 1000 nouveaux films...


Enrichissement v5:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Terminé ! 1000 nouveaux films traités


In [7]:
# === Sauvegarde du dataset FINAL à 4000 films ===

# Convertir les nouvelles données OMDb en DataFrame
df_omdb_5 = pd.DataFrame(omdb_data_5)

# Enrichir les nouveaux films
df_nouveaux_enrichis = df_nouveaux.merge(df_omdb_5, on='imdb_id', how='left')

print(f"Nouveaux films enrichis : {len(df_nouveaux_enrichis)} lignes, {df_nouveaux_enrichis.shape[1]} colonnes")
print(f"Dataset existant : {len(df_enriched_actuel)} lignes, {df_enriched_actuel.shape[1]} colonnes")

# Concaténer
df_final_4000 = pd.concat([df_enriched_actuel, df_nouveaux_enrichis], ignore_index=True)
df_final_4000 = df_final_4000.drop_duplicates(subset=['imdb_id']).reset_index(drop=True)

print(f"\n📊 Dataset final 4000 films : {df_final_4000.shape}")

# Vérification
print(f"\nRemplissage des colonnes OMDb :")
for col in ['plot', 'poster', 'director', 'actors']:
    if col in df_final_4000.columns:
        pct = df_final_4000[col].notna().sum() / len(df_final_4000) * 100
        print(f"  {col} : {pct:.1f}%")
    else:
        print(f"  {col} : COLONNE INTROUVABLE")

# Sauvegarder
df_final_4000.to_csv(DATA_CLEAN + 'imdb_4000_enriched.csv', index=False)
print(f"\n🎉 Sauvegardé : imdb_4000_enriched.csv")
print(f"🎉 ENRICHISSEMENT TERMINÉ ! Tu as 4000 films cultes prêts pour le ML !")

Nouveaux films enrichis : 1000 lignes, 51 colonnes
Dataset existant : 3000 lignes, 51 colonnes

📊 Dataset final 4000 films : (4000, 51)

Remplissage des colonnes OMDb :
  plot : 99.9%
  poster : 99.9%
  director : 99.9%
  actors : 99.9%

🎉 Sauvegardé : imdb_4000_enriched.csv
🎉 ENRICHISSEMENT TERMINÉ ! Tu as 4000 films cultes prêts pour le ML !
